# Notebook 1: QSARmil multi-conformer modelling

This notebook introduces the high-level API for building multi-conformer models with **QSARmil**. The API is designed to simplify benchmarking QSARmil against alternative approaches without requiring detailed knowledge of the underlying modelling configuration. As a result, the workflow remains concise and easy to follow. You can adapt the code below to your own dataset and run the complete **QSARmil** modelling pipeline with minimal modifications.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

### 1. Data load

As an example, we use a publicly available and easily accessible collection of molecular bioactivity datasets introduced in the paper by:

> Van Tilborg, Derek, Alisa Alenicheva, and Francesca Grisoni. "Exposing the limitations of molecular machine learning with activity cliffs." Journal of chemical information and modeling 62.23 (2022): 5938-5951.

From this collection, we select one dataset for demonstration purposes.

In [2]:
# load data
url = "https://raw.githubusercontent.com/molML/MoleculeACE/main/MoleculeACE/Data/benchmark_data/CHEMBL2034_Ki.csv"
df_ace = pd.read_csv(url)

# train/test split
df_train = df_ace[df_ace["split"] == "train"][["smiles", "y"]]
df_test = df_ace[df_ace["split"] == "test"][["smiles", "y"]]

### 2. Data validation

The data validation step is straightforward but includes an additional requirement compared to standard 2D modelling pipelines. In most QSAR workflows, input structures are primarily validated by checking whether their SMILES strings are valid and can be successfully parsed into molecular objects.

In contrast, **QSARmil** introduces an extra validation step: verifying whether a valid 3D structure can be generated for each molecule. Molecules that fail this step cannot be used in the 3D modelling pipeline. As a result, the filtering (removal) rate in QSARmil may be higher than in typical 2D QSAR workflows.

In [3]:
from qsarmil.data.input_data import DataValidator

In [4]:
dvalid = DataValidator(num_cpu=30, verbose=True)

df_train = dvalid.filter_dataframe(df_train)
df_test = dvalid.filter_dataframe(df_test)

No rows removed. All molecules are valid.
No rows removed. All molecules are valid.


### 3. Building model

The multi-conformer model building pipeline consists of several sequential steps:

 - Conformer generation: the maximum number of conformers is defined using the ``num_conf`` parameter.
 - Descriptor calculation: by default, multiple types of 3D molecular descriptors are computed.
 - Model training: several multi-instance learning methods are used to train models. Internal stepwise hyperparameter optimization can be enabled with ``hopt=True`` (note that this increases runtime).
 - Consensus search: once multiple multi-conformer models are trained, a genetic algorithm is applied to identify an optimal consensus model.

Additional parameters include ``task`` (regression or classification), ``output_folder`` (directory for storing predictions from individual models), and ``verbose`` (controls the level of pipeline progress output).

In [5]:
from qsarmil.meta import MultiConformerModel

In [6]:
model = MultiConformerModel(num_conf=10, hopt=False, task="regression", output_folder="mcf", verbose=True)
df_pred = model.run_predict(df_train, df_test)

Generating conformers: 478/478
Generating conformers: 120/120
Generating conformers: 152/152
[1/81] Running model: RDKitGEOM|MeanBagWrapperRidgeRegressor
  > Finished in 0.00 min | Memory usage: 0.899 GB
[2/81] Running model: RDKitGEOM|MeanBagWrapperLinearSVRRegressor
  > Finished in 0.00 min | Memory usage: 0.899 GB
[3/81] Running model: RDKitGEOM|MeanBagWrapperXGBRegressor
  > Finished in 0.01 min | Memory usage: 0.899 GB
[4/81] Running model: RDKitGEOM|MeanBagWrapperMLPRegressor
  > Finished in 0.03 min | Memory usage: 0.900 GB
[5/81] Running model: RDKitGEOM|MeanInstanceWrapperRidgeRegressor
  > Finished in 0.01 min | Memory usage: 0.900 GB
[6/81] Running model: RDKitGEOM|MeanInstanceWrapperLinearSVRRegressor
  > Finished in 0.00 min | Memory usage: 0.900 GB
[7/81] Running model: RDKitGEOM|MeanInstanceWrapperXGBRegressor
  > Finished in 0.02 min | Memory usage: 0.900 GB
[8/81] Running model: RDKitGEOM|MeanInstanceWrapperMLPRegressor
  > Finished in 0.30 min | Memory usage: 0.900 GB

In [7]:
r2_score(df_test["y"], df_pred["pred"])

0.5843936161975568